# Unified Zeek TSV Preprocessing Notebook

This notebook standardizes preprocessing for Zeek-generated TSV files.

In [ ]:
import sys

sys.path.insert(0, '..')

from __future__ import annotations
import numpy as np
import pandas as pd

from utils.preprocessing import (
    FEATURES
)

from sklearn.model_selection import train_test_split

In [ ]:
def normalize_label_name(label: str) -> str:
    label = str(label).strip()
    return (
        label.replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
        .upper()
    )


def compute_and_add_time_elapsed(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values(["id.orig_h", "id.resp_h", "ts"]).reset_index(drop=True)
    df["time_elapsed"] = (
        df.groupby(["id.orig_h", "id.resp_h"])["ts"]
        .diff()
        .fillna(999999.0)
    )
    return df


def compute_valid_tcp_handshake_feature(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    proto = df["proto"].astype(str).str.lower()
    history = df["history"].astype(str)
    conn_state = df["conn_state"].astype(str).str.upper()

    proto_ok = proto.eq("tcp")
    history_ok = history.str.contains(r"S.*h.*A", regex=True, na=False)
    state_ok = conn_state.isin(["S1", "SF"])

    df["valid_tcp_handshake_feature"] = (proto_ok & history_ok & state_ok).astype(float)
    return df


def add_shared_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    numeric_cols = [
        "ts",
        "duration",
        "orig_bytes",
        "resp_bytes",
        "missed_bytes",
        "orig_pkts",
        "orig_ip_bytes",
        "resp_pkts",
        "resp_ip_bytes",
        "id.resp_p",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    existing_numeric = [c for c in numeric_cols if c in df.columns]
    if existing_numeric:
        df[existing_numeric] = df[existing_numeric].replace([np.inf, -np.inf], np.nan)
        df[existing_numeric] = df[existing_numeric].fillna(0.0)

    df = compute_and_add_time_elapsed(df)
    df = compute_valid_tcp_handshake_feature(df)

    duration_safe = df["duration"].copy()

    duration_safe = duration_safe.replace([np.inf, -np.inf], np.nan)
    duration_safe = duration_safe.fillna(0.0)
    duration_safe = duration_safe.mask(duration_safe <= 0, 1e-6)

    df["orig_pkt_rate"] = df["orig_pkts"] / duration_safe
    df["orig_byte_rate"] = df["orig_bytes"] / duration_safe
    df["flood_rate"] = df["orig_bytes"] / duration_safe
    df["pkt_asymmetry"] = df["orig_pkts"] / (df["resp_pkts"] + 1.0)
    df["byte_asymmetry"] = df["orig_bytes"] / (df["resp_bytes"] + 1.0)
    df["is_http"] = df["service"].astype(str).str.lower().eq("http").astype(float)

    engineered_cols = [
        "orig_pkt_rate",
        "orig_byte_rate",
        "pkt_asymmetry",
        "byte_asymmetry",
        "time_elapsed",
        "flood_rate",
        "valid_tcp_handshake_feature",
        "is_http",
    ]

    for col in engineered_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            df[col] = df[col].fillna(0.0)

    return df


def add_portscan_attack_features(df: pd.DataFrame, window_seconds: float = 5.0) -> pd.DataFrame:
    df = df.copy()

    required = {"ts", "id.orig_h", "id.resp_p", "orig_pkts", "duration", "conn_state"}
    if not required.issubset(df.columns):
        for col in ["uniq_dst_ports", "pkts_per_port", "scan_duration", "fail_ratio"]:
            df[col] = 0.0
        return df

    df["window_id"] = (df["ts"] // window_seconds).astype(int)
    group_cols = ["id.orig_h", "window_id"]

    agg = df.groupby(group_cols).agg(
        uniq_dst_ports=("id.resp_p", "nunique"),
        total_orig_pkts=("orig_pkts", "sum"),
        scan_duration=("duration", "max"),
        total_flows=("id.orig_h", "size"),
    ).reset_index()

    FAILED_STATES = {"S0", "REJ", "RSTO", "RSTR", "RSTOS0", "RSTRH", "SH", "SHR"}
    df["is_failed_conn"] = df["conn_state"].astype(str).isin(FAILED_STATES).astype(int)

    fail_agg = df.groupby(group_cols).agg(
        failed_flows=("is_failed_conn", "sum"),
    ).reset_index()

    agg = agg.merge(fail_agg, on=group_cols, how="left")
    agg["failed_flows"] = agg["failed_flows"].fillna(0.0)

    agg["pkts_per_port"] = agg["total_orig_pkts"] / agg["uniq_dst_ports"]
    agg["fail_ratio"] = agg["failed_flows"] / agg["total_flows"]

    df = df.merge(
        agg[
            [
                "id.orig_h",
                "window_id",
                "uniq_dst_ports",
                "pkts_per_port",
                "scan_duration",
                "fail_ratio",
            ]
        ],
        on=["id.orig_h", "window_id"],
        how="left",
    )

    for col in ["uniq_dst_ports", "pkts_per_port", "scan_duration", "fail_ratio"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    return df

def add_ddos_udp_flood_features(
    df: pd.DataFrame,
    window_seconds: float = 5.0,
) -> pd.DataFrame:
    df = df.copy()

    df["is_udp"] = df["proto"].astype(str).str.lower().eq("udp").astype(int)
    df["window_id"] = (df["ts"] // window_seconds).astype(int)

    group_cols = ["id.resp_h", "window_id"]
    udp_df = df[df["is_udp"] == 1]

    agg = udp_df.groupby(group_cols).agg(
        udp_conn_count=("id.orig_h", "size"),
        udp_packets=("orig_pkts", "sum"),
        unique_src_ips=("id.orig_h", "nunique"),
    ).reset_index()

    # Use window duration instead of flow duration
    agg["udp_rate"] = agg["udp_packets"] / window_seconds

    df = df.merge(
        agg,
        on=["id.resp_h", "window_id"],
        how="left",
    )

    for col in [
        "udp_conn_count",
        "udp_packets",
        "udp_rate",
        "unique_src_ips",
    ]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    return df

def add_ddos_syn_flood_features(df: pd.DataFrame, window_seconds: float = 5.0) -> pd.DataFrame:
    df = df.copy()

    df["window_id"] = (df["ts"] // window_seconds).astype(int)

    tcp_df = df[df["proto"].astype(str).str.lower() == "tcp"].copy()

    tcp_df["is_syn"] = tcp_df["history"].astype(str).str.contains("S", regex=False).astype(int)
    tcp_df["is_ack"] = tcp_df["history"].astype(str).str.contains("A", regex=False).astype(int)

    group_cols = ["id.resp_h", "window_id"]

    agg = tcp_df.groupby(group_cols).agg(
        syn_conn_count=("id.orig_h", "size"),
        syn_count=("is_syn", "sum"),
        ack_count=("is_ack", "sum"),
        source_ip_count=("id.orig_h", "nunique"),
    ).reset_index()

    agg["half_open_count"] = (agg["syn_count"] - agg["ack_count"]).clip(lower=0)
    agg["syn_duration"] = window_seconds
    agg["syn_rate"] = agg["syn_count"] / window_seconds

    df = df.merge(
        agg[
            [
                "id.resp_h",
                "window_id",
                "syn_duration",
                "syn_conn_count",
                "syn_count",
                "syn_rate",
                "half_open_count",
                "source_ip_count",
            ]
        ],
        on=["id.resp_h", "window_id"],
        how="left",
    )

    for col in [
        "syn_duration",
        "syn_conn_count",
        "syn_count",
        "syn_rate",
        "half_open_count",
        "source_ip_count",
    ]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    return df


In [ ]:
def load_and_preprocess_data(datapath: str, target_labels: list[str] , window_seconds: float = 5.0) -> pd.DataFrame:
    print(f"Loading data from: {datapath}")
    df = pd.read_csv(datapath, on_bad_lines="skip", delimiter="\t")
    print("Raw shape:", df.shape)
    df.columns = df.columns.str.strip()

    df.drop_duplicates(inplace=True)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df["label"] = df["label"].astype(str).map(normalize_label_name)
    target_labels = [normalize_label_name(x) for x in target_labels]
    df["label"] = df["label"].apply(lambda x: "ATTACK" if x not in target_labels else x)
    df = df[df["label"].isin(target_labels)].copy()

    df = add_shared_engineered_features(df)
    df = add_portscan_attack_features(df, window_seconds=window_seconds)
    df = add_ddos_udp_flood_features(df, window_seconds=window_seconds)
    df = add_ddos_syn_flood_features(df, window_seconds=window_seconds)

    return df

In [ ]:
WINDOW_SECONDS = 5.0
TARGET_LABELS = ["BENIGN", "DOS_HTTP_FLOOD", "PORTSCAN", "DDOS_UDP_FLOOD", "DDOS_SYN_FLOOD", "ATTACK"]

def load_cicids2017_data() -> pd.DataFrame:
    print("Loading CICIDS2017 data...")
    df_cicids2017_wednesday = load_and_preprocess_data(
        datapath="../data/CICIDS2017/wednesday_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    df_cicids2017_friday = load_and_preprocess_data(
        datapath="../data/CICIDS2017/friday_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    res = pd.concat([df_cicids2017_wednesday, df_cicids2017_friday], ignore_index=True)
    print("Processed shape:", res.shape)
    print(res["label"].value_counts())
    return res

def load_cicddos2019_data() -> pd.DataFrame:
    print("Loading CICIDS2019 data...")
    df_cicddos2019_1 = load_and_preprocess_data(
        datapath="../data/CICDDoS2019/cicddos2019_first_day_1_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    df_cicddos2019_2 = load_and_preprocess_data(
        datapath="../data/CICDDoS2019/cicddos2019_first_day_2_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    df_cicddos2019_3 = load_and_preprocess_data(
        datapath="../data/CICDDoS2019/cicddos2019_first_day_3_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    df_cicddos2019_4 = load_and_preprocess_data(
        datapath="../data/CICDDoS2019/cicddos2019_first_day_4_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    df_cicddos2019_5 = load_and_preprocess_data(
        datapath="../data/CICDDoS2019/cicddos2019_second_day_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    res = pd.concat([df_cicddos2019_1, df_cicddos2019_2, df_cicddos2019_3, df_cicddos2019_4, df_cicddos2019_5], ignore_index=True)
    print("Processed shape:", res.shape)
    print(res["label"].value_counts())
    return res

def load_ciciot2023_data() -> pd.DataFrame:
    print("Loading CICIoT2023 data...")
    res = load_and_preprocess_data(
        datapath="../data/CICIoT2023/ciciot2023_labeled_conn.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    print("Processed shape:", res.shape)
    if "label" in res.columns:
        print(res["label"].value_counts())
    return res

In [ ]:
def create_combined_dataset(df_cicids2017, df_cicddos2019) -> pd.DataFrame:
    combined_df = pd.concat([df_cicids2017, df_cicddos2019], ignore_index=True)
    print("Combined dataset shape:", combined_df.shape)
    print(combined_df["label"].value_counts())
    return combined_df

In [ ]:
def downsample_label(df: pd.DataFrame, label: str, frac: float) -> pd.DataFrame:
    label_df = df[df["label"] == label]
    df_other = df[df["label"] != label]
    label_df = label_df.sample(frac=frac, random_state=42)
    combined_df = pd.concat([label_df, df_other])
    print(combined_df["label"].value_counts())
    return combined_df

In [ ]:
df_cicids2017 = load_cicids2017_data()

In [ ]:
df_cicddos2019 = load_cicddos2019_data()

In [ ]:
df_cicids_combined = create_combined_dataset(df_cicids2017, df_cicddos2019)

In [ ]:
df_cicids_combined_sampled = downsample_label(df_cicids_combined, label="DDOS_SYN_FLOOD", frac=0.1)
df_cicids_combined_sampled = downsample_label(df_cicids_combined_sampled, label="BENIGN", frac=0.25)

In [ ]:
df_cicids_combined_sampled.to_csv(f"cicids_combined_preprocessed.tsv", sep='\t', index=False, header=True)

In [ ]:
df_ciciot2023 = load_ciciot2023_data()

In [ ]:
df_ciciot2023_sampled = downsample_label(df_ciciot2023, label="DDOS_SYN_FLOOD", frac=0.1)
df_ciciot2023_sampled = downsample_label(df_ciciot2023_sampled, label="ATTACK", frac=0.25)

In [ ]:
df_ciciot2023_sampled.to_csv(f"ciciot2023_preprocessed.tsv", sep='\t', index=False, header=True)

# Validation of data

In [ ]:
df_cicids_combined_sampled.head()

In [ ]:
df_ciciot2023_sampled.head()

# Create balanced datasets

In [2]:
# Load the preprocessed datasets
df_cicids_combined = pd.read_csv("cicids_combined_preprocessed.tsv", on_bad_lines="skip", delimiter="\t")
df_ciciot2023 = pd.read_csv("ciciot2023_preprocessed.tsv", on_bad_lines="skip", delimiter="\t")

In [3]:
X_cicids = df_cicids_combined[FEATURES].copy()
y_cicids = df_cicids_combined["label"].copy()
X_ciciot2023 = df_ciciot2023[FEATURES].copy()
y_ciciot2023 = df_ciciot2023["label"].copy()

In [4]:
X_train_cicids, X_test_cicids, y_train_cicids, y_test_cicids = train_test_split(
    X_cicids,
    y_cicids,
    test_size=0.3,
    stratify=y_cicids,
    random_state=42,
)

In [26]:
X_train_ciciot2023, X_test_ciciot2023, y_train_ciciot2023, y_test_ciciot2023 = train_test_split(
    X_ciciot2023,
    y_ciciot2023,
    test_size=0.3,
    stratify=y_ciciot2023,
    random_state=42,
)

In [20]:
def balance_df(df: pd.DataFrame) -> pd.DataFrame:
    min_count = df["label"].value_counts().min()
    print(f"Balancing dataset to {min_count} rows per class")

    sampled_idx = (
        df.groupby("label")
        .sample(n=min_count, random_state=42)
        .index
    )

    balanced_df = (
        df.loc[sampled_idx]
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    return balanced_df

In [27]:
X_train_cicids["label"] = y_train_cicids.values
X_train_ciciot2023["label"] = y_train_ciciot2023.values

In [28]:
train_cicids_bal = balance_df(X_train_cicids)
train_ciciot2023_bal = balance_df(X_train_ciciot2023)

Balancing dataset to 107824 rows per class
Balancing dataset to 151573 rows per class


In [25]:
train_cicids_bal.to_csv(f"cicids_combined_balanced_train.tsv", sep='\t', index=False, header=True)

In [29]:
train_ciciot2023_bal.to_csv(f"ciciot2023_balanced_train.tsv", sep='\t', index=False, header=True)

In [30]:
# Save test sets as well for later evaluation
X_test_cicids["label"] = y_test_cicids.values
X_test_ciciot2023["label"] = y_test_ciciot2023.values
X_test_cicids.to_csv(f"cicids_combined_test.tsv", sep='\t', index=False, header=True)
X_test_ciciot2023.to_csv(f"ciciot2023_test.tsv", sep='\t', index=False, header=True)